### Machine Translation


Login hugging face

In [ ]:
from huggingface_hub import login
login()

### Configs

In [ ]:
tl_parallel_corporas = [
    #("dataset/tl-bcl-corpora.csv","bcl"),
    #("dataset/tl-chv-corpora.csv","chv"),
    #("dataset/tl-btn-corpora.csv","btn"),
    ("dataset/tl-ceb-corpora.csv","ceb"),
    # ("dataset/tl-en-corpora.csv","en"),
    #("dataset/tl-ilk-corpora.csv","ilo"),
    ("dataset/tl-ilg-corpora.csv","ilg"),
    #("dataset/tl-ivt-corpora.csv","ivt"),
    #("dataset/tl-kry-corpora.csv","kry"),
    #("dataset/tl-mbo-corpora.csv","mbo"),
    #("dataset/tl-mbyo-corpora.csv","mbyo"),
    #("dataset/tl-pam-corpora.csv","pam"),
    #("dataset/tl-pgn-corpora.csv","pgn"),
    #("dataset/tl-rblo-corpora.csv","rblo"),
    #("dataset/tl-sbl-corpora.csv","sbl"),
    # ("dataset/tl-spa-corpora.csv","spa"),
    #("dataset/tl-tsg-corpora.csv","tsg"),
    #("dataset/tl-wry-corpora.csv","wry"),
    #("dataset/tl-ykn-corpora.csv","ykn"),
    #("dataset/tl-ymi-corpora.csv","ymi")]
    ]

In [ ]:
from transformers import ( BitsAndBytesConfig, M2M100Tokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, M2M100ForConditionalGeneration)
import torch
import pandas as pd
from peft import LoraConfig, get_peft_model

MODEL_NAME = "models/extended_lang_m2m100_1.2B"
# configs
# quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    # use nf4 instead of fp4 for quantization performance
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# lora config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

### Initializing Model

initialize model for quantization and LORA for lower VRAM consumption


In [ ]:
# model config
# model_name = "facebook/m2m100_1.2B"
model_name = "./models/extended_lang_m2m100_1.2B_tl-ceb"
#model_name = "./models/extended_lang_m2m100_1.2B_tl-ilg"

tokenizer_name = "./tokenizers/extended_m2m100tokenizer"
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

model = get_peft_model(model,lora_config)
model.print_trainable_parameters()


### Initialized Tokenizer and Import Datasets
Lets finetune the model to support tagalog to the other languages in the avaialble corpora.


In [ ]:
import datasets
tokenizer = M2M100Tokenizer.from_pretrained(tokenizer_name,src_lang="tl")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(batch):
    
    model_inputs = tokenizer(
        batch["language1_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    labels = tokenizer(
        batch["language2_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs





### Training


In [ ]:
from transformers import Seq2SeqTrainer
import numpy as np
import evaluate

sacrebleu = evaluate.load("sacrebleu")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]

    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}

    return result


In [ ]:
import pandas as pd
from transformers import Seq2SeqTrainingArguments
for df_path, lang_code in tl_parallel_corporas:


    training_args = Seq2SeqTrainingArguments(
        output_dir=MODEL_NAME+"_tl-"+lang_code,
        per_device_train_batch_size=5,
        per_device_eval_batch_size=12,
        num_train_epochs=3,
        learning_rate=3e-5,
        logging_steps=1000,
        save_steps=2000,
        save_total_limit=2,
        eval_steps=5000,
        eval_strategy="steps",
        gradient_accumulation_steps=2,
        predict_with_generate=True,
        fp16=False,
        bf16=True,
        report_to="none"
    )
    print(lang_code)
    df = pd.read_csv(df_path)
    tokenizer.tgt_lang = lang_code
    train_df = df.sample(frac=0.8, random_state=42)
    eval_df = df[~df.index.isin(train_df.index)]
    train_df = train_df.reset_index(drop=True)

    training_dataset = datasets.Dataset.from_pandas(train_df)
    evaluation_dataset = datasets.Dataset.from_pandas(eval_df)
    train_dataset_processed = training_dataset.map(preprocess, batched=True, remove_columns=['language1_text', 'language2_text','Unnamed: 0'])
    eval_dataset_processed = evaluation_dataset.map(preprocess, batched=True, remove_columns=['language1_text', 'language2_text','Unnamed: 0'])

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding="longest",  # pads to the longest sequence in the batch
        return_tensors="pt" # converts lists -> PyTorch tensors
    )
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset_processed,
        eval_dataset=eval_dataset_processed,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=data_collator
    )


    batch = next(iter(trainer.get_train_dataloader()))
    batch = {k: v.to(model.device) for k, v in batch.items()}
    
    output = model(**batch)
    loss = output.loss
    print("-=============-")
    print("tl "+"to "+lang_code)
    print("loss:", loss)
    print("requires_grad:", loss.requires_grad)  # should be True
    print("grad_fn:", loss.grad_fn)  
    trainer.evaluate()
